# BIOINFORMATION ANALYSIS STRATEGY - MENTORING PROJECT

**Kouao Jean Marc NOE**

**Tutor : Dr Pakyendou Estel NAME** 

**Supervisor : Prof Fidèle TIENDREBEOGO & Dr Ezechiel Bionimian TIBIRI**

> This Jupyter is inspired by the model created by E.P.NAME

## Tables of contents 

[PRACTICE I - Acquisition and reorganization of data from sequencing](#practiceI)

[Practice II - Quality control of samples](PraticeII)
- [`nanoplot` NanoPlot](#nanoplot)
- [`NanoComp` NanoComp](#NanoComp)

## Practice I - Acquisition and reorganization of data from sequencing

In [ ]:
# Connect to the cluster on your personal device
ssh noe@160.120.108.164

In [ ]:
# Connect via node02 of the cluster
srun -c 1 -p short --nodelist=node02 --pty bash -i

In [ ]:
# Check the directory you are in 
pwd

In [ ]:
# Navigate to the scratch/noe working folder and create the sugarcane_projet working folder.
cd /scratch/noe
mkdir sugarcane_projet
cd sugarcane_projet # se deplacer dans le dossier 

In [ ]:
# Show folder contents
ls -l 

In [ ]:
# Let's create the project subdirectories: banks, raw data, logs, results, scripts
mkdir banks logs raw_data results scripts

In [ ]:
# Navigate to the raw_data folder to copy the fastq sequences of the samples (Metagenomics)
cd raw_data 
cp cp -r  Metagenomics /scratch/noe/sugarcane_projet/raw_data/

In [ ]:
# Reorganizing files (fastq) by samples (barcodes {75...85}) using this bash script

#!/bin/bash

# Dossier source et destination

SOURCE_DIR="/scratch/noe/sugarcane_projet/raw_data/Metagenomics/sugarecane_demux"
Dest_dir="/scratch/noe/sugarcane_projet/raw_data"

# Créer le dossier de destination pour chaque barcodes
mkdir -p "$Dest_dir"

# Boucle sur les barcodes (75 à 85)
for b in {75..85}; do
    # Créer le dossier spécifique pour le barcode (ex: barcode75)
    TARGET_DIR="$Dest_dir/barcode${b}"
    mkdir -p "$TARGET_DIR"

    echo "Traitement du barcode ${b}..."
# Trouvons tous les fichiers fastq des barcodes dans le dossier source puis copions ses fichiers dans 
    # dans le dossier de destination dans les barcodes correspondantes
    find "$SOURCE_DIR" -type f -name "*barcode${b}*.fastq" | while read -r file; do
        # On extrait le nom du run (ex: AQN321)
      
        BASE_NAME=$(basename "$file")

        # Copie et renommage du fichier 
        cp "$file" "$TARGET_DIR/${BASE_NAME}"
    done
done





In [ ]:
# Merging of the FASTQ files by sample because we have 3 FASTQ files per sample (barcodes)

cd raw_data # navigate within the folder

#!/bin/bash
# Dossier où seront stockés les fichiers fusionnés
mkdir -p final_merged

for b in {75..85}; do
    echo "Fusion en cours pour le barcode ${b}..."
    
    # On vérifie si le dossier du barcode existe pour éviter les erreurs
    if [ -d "barcode${b}" ]; then
        # On fusionne tous les fichiers du dossier dans un seul fichier final
        # Le '>' crée le fichier ou l'écrase s'il existe déjà
        cat "barcode${b}/"*.fastq > "final_merged/barcode${b}_merged.fastq"
        
        echo "Fichier barcode${b}_merged.fastq créé avec succès."
    else
        echo "Attention : Le dossier barcode${b} est introuvable."
    fi
done



In [ ]:
# Let's move the merged files manually into a new folder, a subfolder of raw_data.

mkdir raw_data/merged_fastq
mv barcode75_merged.fastq ../merged_fastq/Barcodes75.fastq # puis pareille avec les autres fichiers



### PRACTICE II -  Control quality of samples

In [ ]:
# Let's create a directory called 1_QC in **/results** in which the quality control results will be stored.

mkdir /results/1_QC

In [ ]:
# Let's navigate to the scripts directory and create the script to launch the quality control.
cd scripts/
nano QC.sh 

#!/bin/bash


# Definissons les variables

RAW_DATA="/scratch/noe/sugarcane_projet/raw_data/merged_fastq"
DIR_QC="/scratch/noe/sugarcane_projet/results/1_QC"

mkdir -p "$DIR_QC"


# Telechargeons des modules
echo "chargement des modules"
module load bioinfo-wave 
module load  nanoplot/1.41.3
module load NanoComp/1.23.1

for file in "${RAW_DATA}"/*.fastq; do
  name=$(basename "$file" .fastq)
  echo "Debut d'analyse de $name"
  # controle qualité des reads
  echo "QC de $name"
  NanoPlot -t "$SLURM_CPUS_PER_TASK" --fastq "$file" -o "$DIR_QC/${name}"
done


# Comparaison QC globale (NanoComp)
echo "Comparaison QC des 11 échantillons"

NanoComp -t "$SLURM_CPUS_PER_TASK" --fastq "$RAW_DATA"/*.fastq -o "$DIR_QC/all_samples_QC"

echo "Analyse terminée !"
